# LeetCode 283 — Move Zeroes bằng Python

**Link đề:** https://leetcode.com/problems/move-zeroes/
**Độ khó:** Easy · **Chủ đề:** Array, Two Pointers

```python
class Solution:
    def moveZeroes(self, nums: list[int]) -> None:
        """
        Do not return anything, modify nums in-place instead.
        """
        ...
```

## Mục tiêu học
1. Hiểu **in-place** (sửa tại chỗ) khác gì với "tạo list mới", và vì sao `nums = ...` **không** sửa được list của người gọi.
2. Làm cách dùng mảng phụ trước, rồi rút ra kỹ thuật **hai con trỏ đọc/ghi** (read/write pointer).
3. Nắm hai lời giải `O(n)`, `O(1)`: "ghi đè rồi lấp 0" và "hoán đổi" (swap) — đáp ứng follow-up giảm số thao tác.
4. Giữ đúng **thứ tự tương đối** của các phần tử khác 0 (tính ổn định — stable).
5. Trace từng bước, chứng minh bằng invariant, và chuyển giao sang các bài cùng khuôn (27, 26, 905).

**Cách học:** đọc markdown → đoán output → chạy cell → so sánh. Cell TODO chỉ định nghĩa hàm, không tự chạy.

## 1. Đề bài

Cho mảng số nguyên `nums`. Dời **tất cả số 0** về **cuối mảng**, đồng thời **giữ nguyên thứ tự tương đối** của các số khác 0.

Phải làm **in-place** — không tạo bản sao của mảng. Hàm **không trả về gì** (`None`).

**Ví dụ 1:** `[0,1,0,3,12]` → `[1,3,12,0,0]`

**Ví dụ 2:** `[0]` → `[0]`

**Constraints:** `1 <= nums.length <= 10^4`; `-2^31 <= nums[i] <= 2^31 - 1` (có số âm!).

**Follow up:** Giảm tổng số thao tác (ghi/hoán đổi) xuống mức tối thiểu.

"Thứ tự tương đối" nghĩa là: `1` đứng trước `3`, `3` đứng trước `12` trong input thì output cũng vậy.

In [1]:
nums = [0, 1, 0, 3, 12]
non_zero = [x for x in nums if x != 0]
print("các số khác 0 theo thứ tự:", non_zero)
print("kết quả mong đợi:", non_zero + [0] * (len(nums) - len(non_zero)))

các số khác 0 theo thứ tự: [1, 3, 12]
kết quả mong đợi: [1, 3, 12, 0, 0]


## 2. Nền tảng Python: tham chiếu, in-place, hoán đổi

### 2.1. Biến là "nhãn dán" lên đối tượng

`a = b` không sao chép list — hai tên cùng trỏ **một** list. Sửa qua `a[i] = ...` thì `b` cũng thấy. Nhưng **gán lại** `a = [...]` chỉ dán nhãn `a` sang list khác, list cũ (mà `b` đang trỏ) **không đổi**.

In [2]:
b_list = [1, 2, 3]
a = b_list
a[0] = 99
print("sửa phần tử qua a -> b_list =", b_list, "| cùng đối tượng?", a is b_list)
a = [7, 7, 7]
print("gán lại a -> b_list vẫn =", b_list, "| cùng đối tượng?", a is b_list)

sửa phần tử qua a -> b_list = [99, 2, 3] | cùng đối tượng? True
gán lại a -> b_list vẫn = [99, 2, 3] | cùng đối tượng? False


### 2.2. Hệ quả khi viết hàm "in-place"

LeetCode gọi hàm của bạn rồi **kiểm tra chính list nó đã truyền vào**. Vì vậy:
- `nums = new_list` trong hàm → vô dụng (chỉ đổi nhãn cục bộ).
- `nums[:] = new_list` → **ghi đè nội dung** của list gốc (slice assignment) — hợp lệ về kết quả, nhưng vẫn tạo list phụ.
- `nums[i] = ...`, `nums[i], nums[j] = nums[j], nums[i]` → sửa tại chỗ thật sự.

In [3]:
def reassign(nums):
    nums = [x for x in nums if x != 0]          # chỉ đổi nhãn cục bộ

def slice_assign(nums):
    nums[:] = [x for x in nums if x != 0] + [0] * nums.count(0)

data = [0, 1, 0, 3, 12]; reassign(data);     print("reassign     ->", data)
data = [0, 1, 0, 3, 12]; slice_assign(data); print("slice_assign ->", data)

reassign     -> [0, 1, 0, 3, 12]
slice_assign -> [1, 3, 12, 0, 0]


### 2.3. Hoán đổi hai phần tử bằng gán đồng thời

`nums[i], nums[j] = nums[j], nums[i]` — Python tính vế phải thành tuple trước, rồi gán lần lượt. Không cần biến tạm.

### 2.4. `id()` để kiểm tra "có còn là list cũ không"

`id(obj)` cho định danh của đối tượng; nếu `id` trước và sau giống nhau thì vẫn là cùng một list.

In [4]:
nums = [10, 20, 30]
before = id(nums)
nums[0], nums[2] = nums[2], nums[0]
print(nums, "| cùng list:", id(nums) == before)

[30, 20, 10] | cùng list: True


## 3. Cách 1 — Mảng phụ (dễ nghĩ nhất)

1. Lấy các số khác 0 theo thứ tự vào list `kept`.
2. Bổ sung đủ số 0.
3. Chép ngược vào `nums` (để kết quả nằm trong list gốc).

Đúng và ổn định, nhưng dùng `O(n)` bộ nhớ phụ — **vi phạm** yêu cầu "không tạo bản sao".

In [5]:
def move_zeroes_extra(nums: list[int]) -> None:
    kept = [x for x in nums if x != 0]
    kept += [0] * (len(nums) - len(kept))
    for i in range(len(nums)):
        nums[i] = kept[i]

data = [0, 1, 0, 3, 12]
move_zeroes_extra(data)
print(data)

[1, 3, 12, 0, 0]


## 4. Cách 2 — "Gặp 0 thì xóa rồi thêm vào cuối" (in-place nhưng chậm)

`nums.remove(0)` xóa số 0 **đầu tiên** và dồn các phần tử phía sau sang trái → `O(n)` mỗi lần. Với nhiều số 0 → `O(n²)`. Ví dụ tốt để thấy "in-place chưa chắc đã nhanh".

In [6]:
def move_zeroes_remove_append(nums: list[int]) -> None:
    zeros = nums.count(0)
    for _ in range(zeros):
        nums.remove(0)        # O(n): dồn phần tử
        nums.append(0)

data = [0, 1, 0, 3, 12]
move_zeroes_remove_append(data)
print(data)

import time
big = [0, 1] * 20000
t0 = time.perf_counter(); move_zeroes_remove_append(big[:]); t1 = time.perf_counter()
print(f"n = {len(big)} với {big.count(0)} số 0: {t1 - t0:.3f}s")

[1, 3, 12, 0, 0]


n = 40000 với 20000 số 0: 1.506s


## 5. Nút thắt và insight — con trỏ **ghi** `w`

Cách 1 thực chất "đọc từ `nums`, ghi các số khác 0 vào chỗ mới". Câu hỏi: **có thể ghi ngay vào `nums`** không?

Có! Dùng hai con trỏ trên **cùng** một mảng:
- `r` (read): duyệt từng phần tử.
- `w` (write): vị trí tiếp theo để đặt một số khác 0.

Vì `w <= r` luôn đúng (ta chỉ ghi khi đã đọc), việc ghi vào `nums[w]` **không bao giờ đè lên** phần tử chưa đọc. Đây là lý do kỹ thuật an toàn khi làm tại chỗ.

```text
nums = [0, 1, 0, 3, 12]
r=0: 0  -> bỏ qua               w=0
r=1: 1  -> nums[0] = 1, w=1
r=2: 0  -> bỏ qua
r=3: 3  -> nums[1] = 3, w=2
r=4: 12 -> nums[2] = 12, w=3
=> [1, 3, 12, 3, 12]  rồi lấp 0 từ w=3: [1, 3, 12, 0, 0]
```

## 6. Cách 3 — Hai con trỏ: ghi đè rồi lấp 0, giảng từng đoạn

**Đoạn 1:** `w = 0`.

**Đoạn 2 — dồn số khác 0 lên đầu:**
```python
for r in range(len(nums)):
    if nums[r] != 0:
        nums[w] = nums[r]
        w += 1
```

**Đoạn 3 — lấp 0 cho phần còn lại:**
```python
for i in range(w, len(nums)):
    nums[i] = 0
```

Số lần ghi = (số phần tử khác 0) + (số số 0) = `n` lần, luôn luôn — kể cả khi mảng **không có số 0 nào** (ghi `nums[i] = nums[i]` vô ích).

In [7]:
def move_zeroes_overwrite(nums: list[int]) -> None:
    w = 0
    for r in range(len(nums)):
        if nums[r] != 0:
            nums[w] = nums[r]
            w += 1
    for i in range(w, len(nums)):
        nums[i] = 0

for data in [[0, 1, 0, 3, 12], [0], [1, 2, 3], [0, 0, 1]]:
    move_zeroes_overwrite(data)
    print(data)

[1, 3, 12, 0, 0]
[0]
[1, 2, 3]
[1, 0, 0]


## 7. Cách 4 — Lời giải chính: hai con trỏ + **hoán đổi** (ít thao tác nhất)

Thay vì ghi đè rồi lấp 0, khi gặp số khác 0 tại `r` ta **đổi chỗ** nó với `nums[w]`. Vì vùng `nums[w..r-1]` chỉ toàn số 0, phép đổi chỗ đưa số khác 0 lên và đẩy một số 0 xuống — **không cần vòng lấp 0**.

Tối ưu thêm cho follow-up: chỉ đổi khi `r != w` (nếu `r == w` thì phần tử đã đúng chỗ, đổi là thừa).

```python
w = 0
for r in range(len(nums)):
    if nums[r] != 0:
        if r != w:
            nums[w], nums[r] = nums[r], nums[w]
        w += 1
```

In [8]:
def move_zeroes_swap(nums: list[int]) -> None:
    w = 0
    for r in range(len(nums)):
        if nums[r] != 0:
            if r != w:
                nums[w], nums[r] = nums[r], nums[w]
            w += 1

for data in [[0, 1, 0, 3, 12], [0], [1, 2, 3], [0, 0, 1], [4, 0, -2, 0, 0, 5]]:
    move_zeroes_swap(data)
    print(data)

[1, 3, 12, 0, 0]
[0]
[1, 2, 3]
[1, 0, 0]
[4, -2, 5, 0, 0, 0]


## 8. TRACE từng bước (cách hoán đổi)

`nums = [0, 1, 0, 3, 12]`:

| r | nums[r] | w trước | hành động | nums sau | w sau |
|---:|---:|---:|---|---|---:|
| 0 | 0 | 0 | bỏ qua | [0, 1, 0, 3, 12] | 0 |
| 1 | 1 | 0 | swap(0, 1) | [1, 0, 0, 3, 12] | 1 |
| 2 | 0 | 1 | bỏ qua | [1, 0, 0, 3, 12] | 1 |
| 3 | 3 | 1 | swap(1, 3) | [1, 3, 0, 0, 12] | 2 |
| 4 | 12 | 2 | swap(2, 4) | [1, 3, 12, 0, 0] | 3 |

Quan sát: luôn có `nums[0..w-1]` = các số khác 0 đã gặp (đúng thứ tự), `nums[w..r]` = toàn 0.

In [9]:
def trace_move_zeroes(nums: list[int]) -> None:
    w = 0
    swaps = 0
    print(f"ban đầu: {nums}")
    for r in range(len(nums)):
        x = nums[r]
        if x != 0:
            if r != w:
                nums[w], nums[r] = nums[r], nums[w]
                swaps += 1
                act = f"swap({w}, {r})"
            else:
                act = "đã đúng chỗ"
            w += 1
        else:
            act = "bỏ qua (0)"
        print(f"  r={r} x={x:>3} | {act:<12} | nums = {nums} | w = {w} | vùng khác 0 = {nums[:w]}")
    print(f"  số lần swap: {swaps}")

trace_move_zeroes([0, 1, 0, 3, 12])
print()
trace_move_zeroes([1, 2, 0, 0, 3])

ban đầu: [0, 1, 0, 3, 12]
  r=0 x=  0 | bỏ qua (0)   | nums = [0, 1, 0, 3, 12] | w = 0 | vùng khác 0 = []
  r=1 x=  1 | swap(0, 1)   | nums = [1, 0, 0, 3, 12] | w = 1 | vùng khác 0 = [1]
  r=2 x=  0 | bỏ qua (0)   | nums = [1, 0, 0, 3, 12] | w = 1 | vùng khác 0 = [1]
  r=3 x=  3 | swap(1, 3)   | nums = [1, 3, 0, 0, 12] | w = 2 | vùng khác 0 = [1, 3]
  r=4 x= 12 | swap(2, 4)   | nums = [1, 3, 12, 0, 0] | w = 3 | vùng khác 0 = [1, 3, 12]
  số lần swap: 3

ban đầu: [1, 2, 0, 0, 3]
  r=0 x=  1 | đã đúng chỗ  | nums = [1, 2, 0, 0, 3] | w = 1 | vùng khác 0 = [1]
  r=1 x=  2 | đã đúng chỗ  | nums = [1, 2, 0, 0, 3] | w = 2 | vùng khác 0 = [1, 2]
  r=2 x=  0 | bỏ qua (0)   | nums = [1, 2, 0, 0, 3] | w = 2 | vùng khác 0 = [1, 2]
  r=3 x=  0 | bỏ qua (0)   | nums = [1, 2, 0, 0, 3] | w = 2 | vùng khác 0 = [1, 2]
  r=4 x=  3 | swap(2, 4)   | nums = [1, 2, 3, 0, 0] | w = 3 | vùng khác 0 = [1, 2, 3]
  số lần swap: 1


### 8.1. Đếm số thao tác ghi: so sánh cách 3 và cách 4

Một phép swap ghi 2 ô. Đo trên hai loại input: ít số 0 và nhiều số 0.

In [10]:
def count_writes_overwrite(nums):
    nums = nums[:]
    writes, w = 0, 0
    for r in range(len(nums)):
        if nums[r] != 0:
            nums[w] = nums[r]; writes += 1; w += 1
    for i in range(w, len(nums)):
        nums[i] = 0; writes += 1
    return writes

def count_writes_swap(nums):
    nums = nums[:]
    writes, w = 0, 0
    for r in range(len(nums)):
        if nums[r] != 0:
            if r != w:
                nums[w], nums[r] = nums[r], nums[w]; writes += 2
            w += 1
    return writes

few_zeros = [1, 2, 3, 4, 5, 6, 7, 8, 9, 0]
many_zeros = [0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
for name, arr in [("ít số 0 (0 ở cuối)", few_zeros), ("nhiều số 0", many_zeros), ("mẫu đề", [0, 1, 0, 3, 12])]:
    print(f"{name:<20}: ghi đè+lấp = {count_writes_overwrite(arr):>2} lần ghi | swap = {count_writes_swap(arr):>2} lần ghi")

ít số 0 (0 ở cuối)  : ghi đè+lấp = 10 lần ghi | swap =  0 lần ghi
nhiều số 0          : ghi đè+lấp = 10 lần ghi | swap =  2 lần ghi
mẫu đề              : ghi đè+lấp =  5 lần ghi | swap =  6 lần ghi


Nhận xét: swap ghi `2 × (số phần tử khác 0 phải di chuyển)`; ghi đè luôn ghi đúng `n`. Không cách nào thắng trên **mọi** input (mẫu đề: swap 6 > ghi đè 5), nhưng khi số 0 ít hoặc đã nằm gần cuối — tình huống thường gặp — swap gần như **không ghi gì**. Đó là ý của follow-up "giảm tổng số thao tác".

## 9. Code nộp LeetCode

In [11]:
class Solution:
    def moveZeroes(self, nums: list[int]) -> None:
        """
        Do not return anything, modify nums in-place instead.
        """
        w = 0
        for r in range(len(nums)):
            if nums[r] != 0:
                if r != w:
                    nums[w], nums[r] = nums[r], nums[w]
                w += 1

data = [0, 1, 0, 3, 12]
ret = Solution().moveZeroes(data)
print(data, "| giá trị trả về:", ret)

[1, 3, 12, 0, 0] | giá trị trả về: None


## 10. Vì sao đúng? (invariant) + độ phức tạp

**Invariant (đầu mỗi vòng `r`):**
1. `nums[0..w-1]` là **tất cả** số khác 0 trong `nums_gốc[0..r-1]`, **đúng thứ tự**.
2. `nums[w..r-1]` toàn là `0`.
3. `nums[r..n-1]` chưa bị động tới.

**Duy trì:** nếu `nums[r] == 0`, mở rộng vùng 0 → (1)(2)(3) vẫn đúng. Nếu `nums[r] != 0`: swap đưa nó vào vị trí `w` (ngay sau các số khác 0 trước đó → giữ thứ tự), số 0 ở `nums[w]` (nếu `w < r`) chuyển sang vị trí `r` (cuối vùng 0); `w += 1`.

**Kết thúc** (`r = n`): `nums[0..w-1]` là mọi số khác 0 theo thứ tự, `nums[w..n-1]` toàn 0 ⇒ đúng yêu cầu.

| Cách | Time | Space phụ | Số lần ghi | Ghi chú |
|---|---|---|---|---|
| 1. Mảng phụ | `O(n)` | `O(n)` | `n` | vi phạm in-place |
| 2. remove + append | `O(n²)` | `O(1)` | nhiều | chậm |
| 3. Ghi đè + lấp 0 | `O(n)` | `O(1)` | đúng `n` | dễ hiểu |
| 4. Swap khi `r != w` | `O(n)` | `O(1)` | `2 ×` số phần tử khác 0 bị dịch | **tối ưu thao tác** |

### 10.1. Bonus — Cách 5: sắp xếp ổn định với `key`

`list.sort` của Python là **ổn định** (phần tử "bằng nhau" theo key giữ thứ tự). Dùng `key=lambda x: x == 0`: số khác 0 có key `False` (= 0), số 0 có key `True` (= 1) → số 0 dồn về cuối, các số khác giữ thứ tự.

Một dòng, in-place, nhưng `O(n log n)` và ẩn đi kỹ thuật — dùng để **kiểm tra chéo** hơn là để nộp phỏng vấn.

In [12]:
def move_zeroes_sort(nums: list[int]) -> None:
    nums.sort(key=lambda x: x == 0)

data = [0, 1, 0, 3, 12, -5, 0, 7]
move_zeroes_sort(data)
print(data, "| False < True:", False < True)

[1, 3, 12, -5, 7, 0, 0, 0] | False < True: True


## 11. Bộ test

In [13]:
def run(f, arr):
    data = arr[:]
    ident = id(data)
    ret = f(data)
    assert ret is None, "hàm in-place không nên trả về giá trị"
    assert id(data) == ident
    return data

def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<28} PASS {actual}")

f = Solution().moveZeroes
check("mẫu 1", run(f, [0, 1, 0, 3, 12]), [1, 3, 12, 0, 0])
check("mẫu 2: [0]", run(f, [0]), [0])
check("không có số 0", run(f, [1, 2, 3]), [1, 2, 3])
check("toàn số 0", run(f, [0, 0, 0]), [0, 0, 0])
check("0 ở cuối sẵn", run(f, [5, 0]), [5, 0])
check("số âm", run(f, [-1, 0, -2, 0, 3]), [-1, -2, 3, 0, 0])
check("giá trị biên int32", run(f, [0, -2**31, 2**31 - 1]), [-2**31, 2**31 - 1, 0])

mẫu 1                        PASS [1, 3, 12, 0, 0]
mẫu 2: [0]                   PASS [0]
không có số 0                PASS [1, 2, 3]
toàn số 0                    PASS [0, 0, 0]
0 ở cuối sẵn                 PASS [5, 0]
số âm                        PASS [-1, -2, 3, 0, 0]
giá trị biên int32           PASS [-2147483648, 2147483647, 0]


In [14]:
import random
rng = random.Random(283)
funcs = [move_zeroes_extra, move_zeroes_remove_append, move_zeroes_overwrite, move_zeroes_swap,
         move_zeroes_sort, Solution().moveZeroes]
def expected_of(arr):
    nz = [x for x in arr if x != 0]
    return nz + [0] * (len(arr) - len(nz))
for _ in range(2000):
    arr = [rng.choice([0, 0, rng.randint(-9, 9)]) for _ in range(rng.randint(1, 30))]
    exp = expected_of(arr)
    assert all(run(f, arr) == exp for f in funcs), arr
print("5 cách khớp nhau trên 2000 mảng ngẫu nhiên (in-place, trả None)")

5 cách khớp nhau trên 2000 mảng ngẫu nhiên (in-place, trả None)


## 12. Lỗi người mới hay gặp

### Lỗi 1: `return` list mới / gán lại `nums`

In [15]:
class WrongReturn:
    def moveZeroes(self, nums):
        nums = [x for x in nums if x != 0] + [0] * nums.count(0)
        return nums

data = [0, 1, 0, 3, 12]
WrongReturn().moveZeroes(data)
print("list của người gọi sau khi gọi:", data, "<- không đổi!")

list của người gọi sau khi gọi: [0, 1, 0, 3, 12] <- không đổi!


**Sửa:** sửa từng phần tử (`nums[i] = ...`, swap) hoặc ít nhất `nums[:] = ...`.

### Lỗi 2: Xóa phần tử khi đang duyệt bằng `for x in nums`

Duyệt và `remove` trên cùng list làm **lệch chỉ số** → bỏ sót phần tử.

In [16]:
def wrong_remove_while_iter(nums):
    n = len(nums)
    for x in nums:                 # SAI: list co lại trong lúc duyệt -> bỏ sót phần tử
        if x == 0:
            nums.remove(0)
    nums.extend([0] * (n - len(nums)))

data = [0, 0, 1]
wrong_remove_while_iter(data)
print("kết quả:", data, "| đúng: [1, 0, 0]")

kết quả: [0, 1, 0] | đúng: [1, 0, 0]


Khi `remove` phần tử ở vị trí đang duyệt, mọi phần tử sau dịch trái một ô, nhưng iterator vẫn tiến tới chỉ số kế tiếp → phần tử vừa dịch vào bị **bỏ qua**. Hai số 0 liền nhau là đủ để lộ lỗi; thử thêm vài input:

In [17]:
for arr in [[0, 0, 1], [0, 1, 0, 3, 12], [1, 0, 0, 0, 2]]:
    d = arr[:]
    wrong_remove_while_iter(d)
    print(f"{arr} -> {d} | đúng: {expected_of(arr)} | {'OK' if d == expected_of(arr) else 'SAI'}")

[0, 0, 1] -> [0, 1, 0] | đúng: [1, 0, 0] | SAI
[0, 1, 0, 3, 12] -> [1, 3, 12, 0, 0] | đúng: [1, 3, 12, 0, 0] | OK
[1, 0, 0, 0, 2] -> [1, 0, 2, 0, 0] | đúng: [1, 2, 0, 0, 0] | SAI


### Lỗi 3: Swap **số 0** lên trước (đảo điều kiện)

In [18]:
def wrong_condition(nums):
    w = 0
    for r in range(len(nums)):
        if nums[r] == 0:                 # SAI: dồn 0 lên đầu
            nums[w], nums[r] = nums[r], nums[w]
            w += 1

d = [0, 1, 0, 3, 12]; wrong_condition(d); print(d)

[0, 0, 1, 3, 12]


### Lỗi 4: Hai con trỏ từ hai đầu (như partition của quicksort)

Đổi số 0 ở trái với số khác 0 ở phải → số 0 về cuối nhưng **phá thứ tự** các số khác 0.

In [19]:
def wrong_two_ends(nums):
    i, j = 0, len(nums) - 1
    while i < j:
        if nums[i] != 0:
            i += 1
        elif nums[j] == 0:
            j -= 1
        else:
            nums[i], nums[j] = nums[j], nums[i]

d = [0, 1, 0, 3, 12]; wrong_two_ends(d); print(d, "| đúng: [1, 3, 12, 0, 0]")

[12, 1, 3, 0, 0] | đúng: [1, 3, 12, 0, 0]


### Lỗi 5: Quên lấp 0 trong cách ghi đè

In [20]:
def wrong_no_fill(nums):
    w = 0
    for r in range(len(nums)):
        if nums[r] != 0:
            nums[w] = nums[r]
            w += 1                        # thiếu vòng lấp 0

d = [0, 1, 0, 3, 12]; wrong_no_fill(d); print(d, "<- đuôi còn rác 3, 12")

[1, 3, 12, 3, 12] <- đuôi còn rác 3, 12


## 13. Mẹo nhận dạng pattern

- **"In-place", "giữ thứ tự tương đối", "dồn/loại phần tử thỏa điều kiện"** → hai con trỏ **cùng chiều** (read/write, còn gọi fast/slow).
- **Không cần giữ thứ tự** → có thể dùng hai con trỏ **hai đầu** (nhanh hơn về số swap).
- Mảng "phân vùng" thành 2 nhóm theo điều kiện → partition; 3 nhóm → Dutch National Flag (bài 75 Sort Colors).

Bài cùng khuôn: 27 Remove Element, 26 Remove Duplicates from Sorted Array, 80 Remove Duplicates II, 905 Sort Array By Parity, 2460 Apply Operations to an Array.

## 14. Biến thể

### 14.1. Dời **một giá trị bất kỳ** `val` về cuối (khuôn tổng quát)

In [21]:
def move_value_to_end(nums: list[int], val: int) -> None:
    w = 0
    for r in range(len(nums)):
        if nums[r] != val:
            nums[w], nums[r] = nums[r], nums[w]
            w += 1

d = [3, 2, 2, 3, 4, 3]; move_value_to_end(d, 3); print(d)

[2, 2, 4, 3, 3, 3]


### 14.2. Dời số 0 về **đầu**, giữ thứ tự số khác 0

Duyệt **từ phải sang trái**, con trỏ ghi bắt đầu ở cuối.

In [22]:
def move_zeroes_to_front(nums: list[int]) -> None:
    w = len(nums) - 1
    for r in range(len(nums) - 1, -1, -1):     # range(start, stop, step) chạy lùi
        if nums[r] != 0:
            nums[w], nums[r] = nums[r], nums[w]
            w -= 1

d = [0, 1, 0, 3, 12]; move_zeroes_to_front(d); print(d)

[0, 0, 1, 3, 12]


### 14.3. Sort Array By Parity (905): số chẵn lên trước (không cần giữ thứ tự)

In [23]:
def sort_by_parity(nums: list[int]) -> list[int]:
    i, j = 0, len(nums) - 1
    while i < j:
        if nums[i] % 2 == 0:
            i += 1
        elif nums[j] % 2 == 1:
            j -= 1
        else:
            nums[i], nums[j] = nums[j], nums[i]
    return nums

print(sort_by_parity([3, 1, 2, 4]))

[4, 2, 1, 3]


## 15. Bài tập (tự làm — cell chỉ định nghĩa, không gọi)

**Bài 1 — Trace tay:** với `[2, 0, 0, 5, 0, 7]` và cách swap, ghi lại `nums` sau mỗi lần swap.

**Bài 2 — Điền chỗ trống:** cách ghi đè + lấp 0.

**Bài 3 — Remove Element (27):** xóa tại chỗ mọi `val`, trả về `k` = số phần tử còn lại (phần đầu `nums[:k]`).

**Bài 4 — Remove Duplicates from Sorted Array (26):** trả về `k`, `nums[:k]` là các giá trị phân biệt theo thứ tự.

**Bài 5 — Dời số 0 về cuối nhưng trả về số lần swap**, đảm bảo tối thiểu (không swap khi `r == w`).

**Bài 6 — Mini challenge — Sort Colors (75):** mảng chỉ gồm 0, 1, 2; sắp xếp tại chỗ trong **một lượt** (Dutch National Flag, ba con trỏ).

In [24]:
# Bài 1 — TODO: list các trạng thái nums sau mỗi lần swap
predict_states = None
# assert predict_states == <xem đáp án>

In [25]:
def exercise_overwrite(nums: list[int]) -> None:
    w = 0
    # TODO 1: dồn số khác 0 lên đầu
    # TODO 2: lấp 0 từ w tới cuối
    pass

# d = [0, 1, 0, 3, 12]; exercise_overwrite(d); assert d == [1, 3, 12, 0, 0]

In [26]:
def exercise_remove_element(nums: list[int], val: int) -> int:
    # TODO
    pass

# d = [3, 2, 2, 3]; k = exercise_remove_element(d, 3); assert k == 2 and sorted(d[:k]) == [2, 2]

In [27]:
def exercise_remove_duplicates(nums: list[int]) -> int:
    # TODO
    pass

# d = [0, 0, 1, 1, 1, 2, 2, 3, 3, 4]; k = exercise_remove_duplicates(d); assert d[:k] == [0, 1, 2, 3, 4]

In [28]:
def exercise_count_swaps(nums: list[int]) -> int:
    # TODO
    pass

# d = [1, 2, 0, 3]; assert exercise_count_swaps(d) == 1 and d == [1, 2, 3, 0]

In [29]:
def exercise_sort_colors(nums: list[int]) -> None:
    # TODO: low, mid, high
    pass

# d = [2, 0, 2, 1, 1, 0]; exercise_sort_colors(d); assert d == [0, 0, 1, 1, 2, 2]

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | chỉ swap khi `nums[r] != 0` và `r != w` | lần đầu `r = 0, w = 0` không swap | swap đầu tiên là `(1, 3)` |
| 2 | xem mục 6 | `nums[w] = nums[r]; w += 1` | `for i in range(w, len(nums)): nums[i] = 0` |
| 3 | giống khuôn ghi đè | giữ phần tử `!= val` | trả `w` |
| 4 | so với phần tử cuối đã ghi | `if nums[r] != nums[w - 1]` | bắt đầu `w = 1` |
| 5 | đếm trong nhánh `r != w` | trả bộ đếm | |
| 6 | `[0..low-1]` = 0, `[low..mid-1]` = 1, `[high+1..]` = 2 | gặp 0: swap với low, tăng cả hai | gặp 2: swap với high, **không** tăng mid |

## 17. Đáp án tham khảo

In [30]:
answer_states = [[2, 5, 0, 0, 0, 7], [2, 5, 7, 0, 0, 0]]
d, states, w = [2, 0, 0, 5, 0, 7], [], 0
for r in range(len(d)):
    if d[r] != 0:
        if r != w:
            d[w], d[r] = d[r], d[w]
            states.append(d[:])
        w += 1
assert states == answer_states
print("Bài 1:", answer_states)

def answer_overwrite(nums):
    w = 0
    for r in range(len(nums)):
        if nums[r] != 0:
            nums[w] = nums[r]
            w += 1
    for i in range(w, len(nums)):
        nums[i] = 0

def answer_remove_element(nums, val):
    w = 0
    for r in range(len(nums)):
        if nums[r] != val:
            nums[w] = nums[r]
            w += 1
    return w

def answer_remove_duplicates(nums):
    if not nums:
        return 0
    w = 1
    for r in range(1, len(nums)):
        if nums[r] != nums[w - 1]:
            nums[w] = nums[r]
            w += 1
    return w

def answer_count_swaps(nums):
    w = swaps = 0
    for r in range(len(nums)):
        if nums[r] != 0:
            if r != w:
                nums[w], nums[r] = nums[r], nums[w]
                swaps += 1
            w += 1
    return swaps

def answer_sort_colors(nums):
    low, mid, high = 0, 0, len(nums) - 1
    while mid <= high:
        if nums[mid] == 0:
            nums[low], nums[mid] = nums[mid], nums[low]
            low += 1; mid += 1
        elif nums[mid] == 1:
            mid += 1
        else:
            nums[mid], nums[high] = nums[high], nums[mid]
            high -= 1

d = [0, 1, 0, 3, 12]; answer_overwrite(d); assert d == [1, 3, 12, 0, 0]
d = [3, 2, 2, 3]; k = answer_remove_element(d, 3); assert k == 2 and sorted(d[:k]) == [2, 2]
d = [0, 0, 1, 1, 1, 2, 2, 3, 3, 4]; k = answer_remove_duplicates(d); assert d[:k] == [0, 1, 2, 3, 4]
d = [1, 2, 0, 3]; assert answer_count_swaps(d) == 1 and d == [1, 2, 3, 0]
d = [2, 0, 2, 1, 1, 0]; answer_sort_colors(d); assert d == [0, 0, 1, 1, 2, 2]
for _ in range(500):
    arr = [rng.randint(0, 2) for _ in range(rng.randint(0, 20))]
    d = arr[:]; answer_sort_colors(d); assert d == sorted(arr)
print("Tất cả đáp án PASS")

Bài 1: [[2, 5, 0, 0, 0, 7], [2, 5, 7, 0, 0, 0]]
Tất cả đáp án PASS


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def moveZeroes(self, nums: list[int]) -> None:
        w = 0
        for r in range(len(nums)):
            if nums[r] != 0:
                if r != w:
                    nums[w], nums[r] = nums[r], nums[w]
                w += 1
```

**Câu thần chú:** *"r đọc, w ghi; gặp số khác 0 thì đưa về w; vùng giữa w và r luôn toàn 0."*

**Checklist:**
- [ ] Sửa **tại chỗ**, không `return`, không gán lại `nums = ...`.
- [ ] Giữ thứ tự số khác 0 (hai con trỏ **cùng chiều**, không phải hai đầu).
- [ ] Không xóa phần tử khi đang duyệt.
- [ ] Nếu ghi đè, nhớ lấp 0 phần đuôi.
- [ ] Số âm vẫn là "khác 0" — điều kiện là `!= 0`, không phải `> 0`.